# IEEE

In [16]:
# Comment out the following line to run the notebook in CPU mode
import plotly.express as px
import pandas as pd

# Suppress warnings
import warnings
from IPython.display import display
import json
import os
from tqdm import tqdm
from pathlib import Path

warnings.filterwarnings("ignore")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*column_view.*")

root_dir = os.path.dirname(os.getcwd())  # Use current working directory as root_dir
display("Running in root dir: ", os.path.basename(root_dir))

output_dir = Path() / "ancova_ieee"
output_dir.mkdir(parents=True, exist_ok=True)
print("Output directory:", output_dir.absolute())


'Running in root dir: '

'freesurfer-fuzzy'

Output directory: /mnt/lustre/ychatel/living-park/VIP-python-client/example/freesurfer-fuzzy/notebooks/ancova_ieee


In [17]:
def get_cohort_stat():
    df_cohort = pd.read_csv("../cohort_stat.csv")
    print(f"Load cohort stats: {os.path.abspath('../cohort_stat.csv')}")
    columns = [
        "PATNO",
        "first_visit",
        "second_visit",
        "dx_group",
        "SEX",
        "AGE_AT_VISIT",
    ]
    df_cohort["first_visit"] = (
        "sub-"
        + df_cohort["PATNO"].astype(str)
        + "_ses-"
        + df_cohort["EVENT_ID"].astype(str)
    )
    df_cohort["second_visit"] = (
        "sub-"
        + df_cohort["PATNO"].astype(str)
        + "_ses-"
        + df_cohort["NEXT_VISIT"].astype(str)
    )
    # Remove PD-MCI subjects
    df_cohort = df_cohort[df_cohort["dx_group"] != "PD-MCI"]
    print(f"Number of PD-non-MCI subjects: {df_cohort.shape[0]}")
    return df_cohort[columns]


df_cohort = get_cohort_stat()


Load cohort stats: /mnt/lustre/ychatel/living-park/VIP-python-client/example/freesurfer-fuzzy/cohort_stat.csv
Number of PD-non-MCI subjects: 270


# ANCOVA

## Cortical

In [18]:
def read_table(hemi, measure):
    df = pd.read_csv(f"table_ieee/{hemi}.aparc.{measure}.tsv", sep="\t")
    df["hemi"] = hemi
    df.columns = [c.replace(f"{hemi}.", "") for c in df.columns]
    df.columns = [c.replace(f"{hemi}_", "") for c in df.columns]
    df.columns = [c.replace(f"_{measure}", "") for c in df.columns]
    df.rename(columns={f"aparc.{measure}": "first_visit"}, inplace=True)
    return df


def read_measure(measure):
    lh = read_table("lh", measure)
    rh = read_table("rh", measure)
    return pd.concat([lh, rh], axis=0)


def get_baseline_ancova(metric):
    df = read_measure(metric)
    df = df.melt(id_vars=["first_visit", "hemi"], var_name="region", value_name=metric)
    df = pd.merge(df, df_cohort, on="first_visit")
    df = df[["first_visit", "hemi", "region", metric, "dx_group", "AGE_AT_VISIT"]]
    return df


In [19]:
import pingouin as pg


def compute_ancova(measure, clinical_df, force):
    if not force and os.path.exists(f"ancova_ieee/pcorr_{measure}.csv"):
        return pd.read_csv(f"ancova_ieee/pcorr_{measure}.csv")

    df = get_baseline_ancova(measure)
    df = pd.merge(
        df,
        clinical_df,
        left_on="first_visit",
        right_on="first_visit",
        suffixes=("", "_clinical"),
    )
    df = df[
        ["first_visit", "region", measure, "hemi", "dx_group", "AGE_AT_VISIT", "SEX"]
    ]

    ancova_df = pd.DataFrame(columns=["hemisphere", "region", "F", "pval"])
    for hemi in df["hemi"].unique():
        for region in df["region"].unique():
            df_region = df[(df["hemi"] == hemi) & (df["region"] == region)]
            ancova = pg.ancova(
                data=df_region,
                dv=measure,
                between="dx_group",
                covar=["AGE_AT_VISIT", "SEX"],
            )
            (F, pval) = ancova["F"].values[0], ancova["p-unc"].values[0]
            ancova_df.loc[len(ancova_df)] = [hemi, region, F, pval]

    filename = output_dir / f"ancova_baseline_{measure}.csv"
    ancova_df.to_csv(filename, index=False)

    return ancova_df


In [20]:
ancova_volume = compute_ancova("volume", df_cohort, force=True)
ancova_thickness = compute_ancova("thickness", df_cohort, force=True)
ancova_area = compute_ancova("area", df_cohort, force=True)

In [21]:
ancova_volume[ancova_volume["pval"] < 0.05].sort_values("F", ascending=False)

,hemisphere,region,F,pval
61,rh,rostralmiddlefrontal,24.475150,0.000001
37,rh,caudalanteriorcingulate,21.467455,0.000006
34,lh,BrainSegVolNotVent,19.195154,0.000017
70,rh,BrainSegVolNotVent,19.195154,0.000017
46,rh,lateralorbitofrontal,17.332902,0.000042
60,rh,rostralanteriorcingulate,15.527914,0.000104
21,lh,posteriorcingulate,14.594114,0.000166
53,rh,parsorbitalis,13.012982,0.000369
16,lh,parsopercularis,11.652634,0.000741
35,lh,eTIV,11.569986,0.000773


In [22]:
ancova_thickness[ancova_thickness["pval"] < 0.05].sort_values("F", ascending=False)

,hemisphere,region,F,pval
35,lh,BrainSegVolNotVent,19.195154,0.000017
72,rh,BrainSegVolNotVent,19.195154,0.000017
36,lh,eTIV,11.569986,0.000773
73,rh,eTIV,11.569986,0.000773
51,rh,parahippocampal,10.466433,0.001369


In [23]:
ancova_area[ancova_area["pval"] < 0.05].sort_values("F", ascending=False)

,hemisphere,region,F,pval
38,rh,caudalanteriorcingulate,24.125025,0.000002
62,rh,rostralmiddlefrontal,23.820015,0.000002
72,rh,BrainSegVolNotVent,19.195154,0.000017
35,lh,BrainSegVolNotVent,19.195154,0.000017
18,lh,parstriangularis,15.032011,0.000133
71,rh,WhiteSurfArea,14.101153,0.000213
21,lh,posteriorcingulate,13.966403,0.000228
54,rh,parsorbitalis,13.946635,0.000230
16,lh,parsopercularis,13.707017,0.000260
34,lh,WhiteSurfArea,13.278499,0.000323


## Subcortical Volume

In [24]:
df = pd.read_csv("table_ieee/aseg.volume.tsv", sep="\t")
df.rename(columns={"Measure:volume": "first_visit"}, inplace=True)
df = df.melt(id_vars=["first_visit"], var_name="region", value_name="volume")
df = pd.merge(
    df,
    df_cohort,
    left_on="first_visit",
    right_on="first_visit",
    suffixes=("", "_clinical"),
)
df = df[["first_visit", "region", "volume", "dx_group", "AGE_AT_VISIT", "SEX"]]

ancova_subcortical_volume_df = pd.DataFrame(columns=["region", "F", "pval"])
for region in tqdm(df["region"].unique()):
    df_region = df[df["region"] == region]
    ancova = pg.ancova(
        data=df_region, dv="volume", between="dx_group", covar=["AGE_AT_VISIT", "SEX"]
    )
    (F, pval) = ancova["F"].values[0], ancova["p-unc"].values[0]
    ancova_subcortical_volume_df.loc[len(ancova_subcortical_volume_df)] = [
        region,
        F,
        pval,
    ]

filename = output_dir / "ancova_baseline_subcortical_volume.csv"
ancova_subcortical_volume_df.to_csv(filename, index=False)

ancova_subcortical_volume_df[ancova_subcortical_volume_df["pval"] < 0.05].sort_values(
    "F", ascending=False
)

 55%|████████████████████████████████████████████████████████████████████████████████▉                                                                   | 35/64 [00:00<00:00, 80.23it/s]

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 80.12it/s]


,region,F,pval
46,BrainSegVolNotVent,19.195154,0.000017
25,Right-Pallidum,18.308144,0.000026
3,Left-Cerebellum-Cortex,17.802033,0.000034
45,BrainSegVol,16.807146,0.000055
56,SupraTentorialVolNotVent,16.644754,0.000060
10,Brain-Stem,16.230835,0.000073
22,Right-Thalamus,15.426325,0.000109
29,Right-VentralDC,15.037277,0.000133
57,MaskVol,15.025067,0.000134
4,Left-Thalamus,14.873146,0.000144
